<a href="https://colab.research.google.com/github/joabpaiva/Google-Colab/blob/main/System_1_JEV_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><p float="center">

  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>Evaluating LLM Outputs with Jev</font></center>
<center><font size=6>System 1 Classifiers: Politeness Checks and RAG Relevance Filtering</font></center>

### Learning objectives

By the end of this notebook you will be able to:

1. Explain how a **decision model (Jev)** differs from an **LLM-as-a-judge**.
2. Ask Jev **typed questions** (yes/no, graded scale, and category) and read the **probabilities** it returns.
3. Evaluate whether a chatbot reply is **polite** using those questions.
4. Build a **mock RAG pipeline** that checks each retrieved chunk for relevance and drops the noise.
5. Choose and reason about **thresholds** and **error handling** in an evaluation pipeline.

### Prerequisites

- Basic Python (functions, loops, dictionaries)
- A **TypeSafe API key** for Jev (access may require signing up; see the TypeSafe docs)
- A Google Colab account

### How to use this notebook

Run the cells **from top to bottom**. Every code cell is preceded by a text cell that explains what the code does and why. Code is split into small cells, one idea per cell, so you can run and inspect each piece on its own.

# **Problem Statement**

## Business Context

BrightCart is an online retailer that answers customer questions through an AI support chatbot. The chatbot handles questions such as *"How do I get a refund?"* and *"Where is my order?"*, and it answers using a library of company policy documents (a **RAG** system, explained later in this notebook).

Before the chatbot can be trusted with customers, the support team has identified two quality risks:

- **Tone risk.** A reply can be factually correct and still sound dismissive or rude (for example, *"Read the docs. This was already answered."*). A single rude reply can damage a customer relationship, and nobody can read every reply before it is sent.
- **Retrieval risk.** The chatbot retrieves several policy chunks before it answers. Retrievers are imperfect and often return chunks that *look* similar to the question but do not help (for example, a chunk about company history when the customer asked about refunds). Irrelevant chunks can **distract** the language model and lead to wrong answers.

Both risks need an automatic check that is **fast, cheap, and consistent**, because the check runs on every reply and every retrieved chunk.

## Objective

The objective is to build **two automatic quality gates** using a decision model called **Jev**:

1. **A politeness gate.** Every chatbot reply is scored for tone. The score is turned into a clear pass or fail verdict, and also into a finer-grained tone level so that the team can see *how* good or bad the tone is.
2. **A relevance gate for RAG.** Every retrieved chunk is checked against the customer's question. Relevant chunks are kept and noisy chunks are dropped before the language model sees them.

Both gates must handle failures safely, because an evaluation service can time out or return an error, and a broken check must never silently remove real evidence.

## Success Criteria

| Business objective | Technical metric | Threshold | Rationale |
|---|---|---|---|
| Stop rude replies reaching customers | P(polite) from a yes/no question | at least 0.50 to pass | A midpoint threshold is the standard starting point. It is tuned later using human-labelled examples. |
| Understand *how* polite a reply is | Tone level from a 4-level scale question | 0 (rude) to 3 (warm) | The scale separates *polite* from *warm*, which a pass/fail answer cannot do. |
| Give the language model clean evidence | P(relevant) for each retrieved chunk | at least 0.50 to keep | Drops chunks that cannot answer the question while keeping those that can. |
| Never lose real evidence silently | Behaviour when the check fails | Keep the chunk ("fail open") | Losing a useful chunk is usually worse than letting one noisy chunk through. |
| Trust the gates | Agreement with human labels | Measured in the practice exercises | Neither a decision model nor an LLM judge is automatically correct. Always measure on examples you have labelled. |

## Data Description

This notebook uses small hand-written datasets so that the focus stays on the evaluation technique. No files need to be uploaded.

**Dataset A: sample chatbot replies.** Three replies for Part 1: one warm, one rude, and one neutral and factual.

**Dataset B: mock knowledge base.** Five policy chunks about refunds for Part 2. Three are relevant to a refund question and two are deliberate noise (company headquarters and company history).

# **Solution Approach**

The solution is built in two measurable parts. Both parts use the same Jev evaluator and the same error-handling pattern.

1. **Learn the core concepts.** What evaluation is, how a decision model differs from an LLM judge, the three question types, and why probabilities and thresholds matter.

2. **Set up the tools.** Install the library, load the API key safely, and create the evaluator.

3. **Define the typed questions.** One yes/no question, one scale question, and one category question for politeness.

4. **Part 1: Politeness gate.** Score each reply, apply a threshold to turn the probability into a verdict, and study how the threshold changes the results.

5. **Part 2: Relevance gate for RAG.** Retrieve chunks from a mock knowledge base, score every chunk, keep the relevant ones, and build the final prompt for a generator.

6. **Practice.** Exercises that extend both parts, including measuring agreement with your own labels.

Key points to note:

- Jev **does not generate text**. It reads a *state* (the thing being judged) and returns typed answers with probabilities.
- Every answer is a **probability, not a guarantee**. The pipeline decides what to do with it using a threshold.
- Evaluators **return an error object instead of raising an exception**, so the code must check for failure explicitly.

## Why use a decision model (Jev) instead of an LLM judge?

A common way to evaluate text is to ask a second language model to act as a **judge**. A decision model is a different tool for the same job.

| | LLM judge | Decision model (Jev) |
|---|---|---|
| Output | Free text (reasoning plus a label) | Typed answers with a **probability for every option** |
| Typical cost and speed | Higher and slower | Lower and faster |
| Repeatability | Can vary between runs | Designed to be consistent |

Because the checks in this notebook run on **every reply and every chunk**, low cost, low latency, and consistent answers matter more than a long written explanation.

**Important:** Neither approach is automatically correct. Always measure any judge on examples you have labelled yourself.

# **Core Concepts**

## What is "evaluation"?

When an AI system produces text, we need a way to check its quality: *Is it polite? Is it grounded in facts? Did it retrieve useful documents?*

Exact-match tests (for example, checking that the output equals a stored answer) fail for open-ended text, because many different sentences can be equally good. This is why evaluation usually relies on a second model that **judges** the output.

## The three question types

You never ask Jev for free text. You give it a **state** and one or more **typed questions**. The type of the question decides the shape of the answer.

| Question type | You provide | You get back |
|---|---|---|
| `NoulQuestion` | A yes/no question | `.noul`: the probability of "yes" (0 to 1) |
| `ChoiceQuestion` | Options, each with a description | `.choice`, `.probabilities`, `.confidence` |
| `ScoreQuestion` | Ordered levels, lowest first | `.score`, `.probabilities`, `.confidence` |

A few details that are easy to miss:

- A `NoulQuestion` answer contains only the probability. It has no `.confidence` field.
- A `ChoiceQuestion` takes a **dictionary** that maps each option name to a description of what that option means.
- A `ScoreQuestion` takes a **list** of levels ordered from lowest to highest. `.score` is the **expected level**, counted from 0, so it can be fractional (for example 2.4 means "between polite and warm, closer to polite").

## Key idea: probabilities, not certainty

A result of `0.91` means *"the model puts 91% probability on yes"*. It is **not** a guarantee.

Good evaluation pipelines therefore:

- Choose a **threshold** (for example 0.5) that turns a probability into a decision.
- Treat low-confidence answers as candidates for **human review** or a stronger model.
- Decide in advance what happens when the evaluation itself **fails**.

# **Installing and Importing Necessary Libraries and Dependencies**

## Install dependencies

In [ ]:
# Install the grounded-ai library with Jev support; -q keeps the output quiet
!pip install -q "grounded-ai[jev]"

**Note:**

- If Colab asks you to restart the runtime after installation, do so, then run all cells sequentially from the next cell.
- Warnings about package dependencies during installation can be ignored.

## Storing your API key safely

Never paste secret keys directly into notebook code, because notebooks get shared. In Colab:

1. Click the **key icon (Secrets)** in the left sidebar.
2. Add a secret named `TYPESAFE_API_KEY` and paste your key as the value.
3. Turn on **Notebook access** for that secret.

In [ ]:
import os  # standard library module for reading/setting environment variables
from google.colab import userdata  # Colab helper that reads secrets from the sidebar

# Read the secret from Colab and expose it as an environment variable the library looks for
os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")

# Print a confirmation without ever showing the key itself
print("API key loaded:", bool(os.environ.get("TYPESAFE_API_KEY")))

API key loaded: True


## Importing necessary libraries

We import `pandas` for readable tables, the `Evaluator` class that talks to Jev, and the Jev building blocks (input wrapper and question types). We also define a small helper because **evaluators return an error object instead of raising an exception**, so we must check for failure ourselves.

In [ ]:
import pandas as pd  # pandas lets us show results as a readable table
from grounded_ai import Evaluator  # the main class used to create a judge/evaluator
from grounded_ai.backends.jev import (  # Jev-specific building blocks
    JevInput,       # wraps the "state" plus the questions we send to Jev
    NoulQuestion,   # yes/no question type
    ScoreQuestion,  # ordered-scale question type
    ChoiceQuestion, # category question type
    RAG_RELEVANCE,  # ready-made question: can this context answer the query?
)

# Create an evaluator that calls TypeSafe's hosted Jev model
jev = Evaluator("jev/jev-latest")


def is_error(result):
    # Evaluators RETURN an error object instead of raising an exception,
    # so we detect failures by checking for an 'error_code' attribute
    return hasattr(result, "error_code")

# **Part 1: Is This Reply Polite?**

## Scenario

Imagine BrightCart's customer-support chatbot. Before showing a reply to a customer, we want to check its tone.

## Evaluation design

We ask Jev **three kinds of question** about each reply, and we study them **one at a time** before combining them:

1. **`polite`** (yes/no, `NoulQuestion`): a simple pass or fail signal.
2. **`tone`** (4-level scale, `ScoreQuestion`): a finer-grained signal, from rude to warm.
3. **`tone_category`** (choice, `ChoiceQuestion`): a named category that describes the tone.

Using all three is useful. The yes/no answer is easy to threshold, the scale shows *how* good or bad the tone is, and the category gives a readable label.

## Step 3.1: Prepare the data

In [ ]:
# Three sample chatbot replies: one warm, one rude, one neutral/factual
replies = [
    "Thanks for reaching out! I'm sorry about the trouble, and I'll fix it right away.",  # warm
    "Read the docs. This was already answered.",                                          # rude
    "Your order has shipped and should arrive Friday.",                                   # neutral
]

## Step 3.2: Question type 1, Yes/no (`NoulQuestion`)

A `NoulQuestion` asks a yes/no question. Jev returns the **probability that the answer is "yes"**, a number between 0 and 1 (read from `.noul`). It has no `.confidence` field.

We also choose a **threshold**: if P(polite) is at least this value, we label the reply POLITE.

### Define the question

In [ ]:
# Yes/no question: Jev returns the probability that the answer is "yes"
polite_question = NoulQuestion(
    instructions="Is this reply polite and respectful toward the customer?"
)

THRESHOLD = 0.5  # if P(polite) is at least this value, we label the reply POLITE

### Run it on every reply

We loop over the replies, send each one to Jev, and collect the results into a table. Notice the **error check** before we read any answer.

In [ ]:
rows = []  # we will collect one dictionary per reply, then turn it into a table

for reply in replies:  # evaluate each reply one at a time
    # Send the reply (as the "state") plus our single yes/no question to Jev
    result = jev.evaluate(JevInput(state=reply, questions={"polite": polite_question}))

    # If the call failed, record the error and move on to the next reply
    if is_error(result):
        rows.append({"reply": reply, "error": f"{result.error_code}: {result.message}"})
        continue

    p_polite = result.answers["polite"].noul  # probability (0-1) that the reply is polite

    rows.append({
        "reply": reply[:60] + ("..." if len(reply) > 60 else ""),         # shortened text for display
        "P(polite)": round(p_polite, 2),
        "verdict": "POLITE" if p_polite >= THRESHOLD else "NOT POLITE",   # apply our threshold
    })

pd.DataFrame(rows)  # display the results as a table

,reply,P(polite),verdict
0,Thanks for reaching out! I'm sorry about the t...,0.97,POLITE
1,Read the docs. This was already answered.,0.16,NOT POLITE
2,Your order has shipped and should arrive Friday.,0.80,POLITE


### Reading the yes/no results

- **P(polite)** close to 1 means Jev strongly believes the reply is polite. Close to 0 means it strongly believes the reply is not.
- **verdict** is simply P(polite) compared with `THRESHOLD`.
- The yes/no question tells us *whether* a reply passes, but not *how* polite it is. The neutral shipping message is the case to watch: it is not rude, but it is not warm either.

## Step 3.3: Question type 2, Graded scale (`ScoreQuestion`)

A `ScoreQuestion` rates the reply against **ordered levels**. You list the levels from **lowest to highest quality**. Jev returns:

- `.score`: the **expected level**, counted from 0. It can be fractional (2.4 means "between polite and warm, closer to polite").
- `.probabilities`: the probability of each level.
- `.confidence`: how tightly the probability clusters on the scale.

### Define the question

In [ ]:
# Scale question: levels are listed from LOWEST to HIGHEST quality
tone_question = ScoreQuestion(
    instructions="How courteous is the tone of this reply?",
    criteria=[
        "rude or dismissive",   # level 0
        "neutral or curt",      # level 1
        "polite",               # level 2
        "warm and courteous",   # level 3
    ],
)

### Run it on every reply

In [ ]:
rows = []

for reply in replies:
    result = jev.evaluate(JevInput(state=reply, questions={"tone": tone_question}))

    if is_error(result):
        rows.append({"reply": reply, "error": f"{result.error_code}: {result.message}"})
        continue

    answer = result.answers["tone"]  # the answer object for our scale question

    rows.append({
        "reply": reply[:60] + ("..." if len(reply) > 60 else ""),
        "tone_score": round(answer.score, 2),            # expected level: 0 = rude ... 3 = warm
        "tone_confidence": round(answer.confidence, 2),  # how confident Jev is in that level
    })

pd.DataFrame(rows)

,reply,tone_score,tone_confidence
0,Thanks for reaching out! I'm sorry about the t...,3.00,1.00
1,Read the docs. This was already answered.,0.17,0.84
2,Your order has shipped and should arrive Friday.,1.26,0.74


### Reading the scale results

- **tone_score** places each reply on our scale (0 = rude, 3 = warm).
- **tone_confidence** shows how sure Jev is. Low values are good candidates for human review.
- Compare this table with the yes/no table. The scale separates *polite* from *warm*, which the yes/no answer cannot do. The neutral reply should land in the middle of the scale.

## Step 3.4: Question type 3, Category (`ChoiceQuestion`)

A `ChoiceQuestion` asks Jev to pick **one option** from a named set. You provide a **dictionary** that maps each option name to a short description of what it means. Jev returns:

- `.choice`: the option it picked.
- `.probabilities`: the probability of every option.
- `.confidence`: how concentrated the probability is (0 means spread evenly, 1 means certain).

### Define the question

In [ ]:
# Choice question: each option name maps to a description of what it means
tone_category_question = ChoiceQuestion(
    instructions="Which category best describes the tone of this reply?",
    criteria={
        "rude":    "the reply is dismissive or disrespectful",
        "neutral": "the reply is factual with no courtesy signals such as thanks or apology",
        "polite":  "the reply is respectful and courteous",
        "warm":    "the reply is friendly and shows empathy for the customer",
    },
)

### Run it on every reply

In [ ]:
rows = []

for reply in replies:
    result = jev.evaluate(JevInput(state=reply, questions={"tone_category": tone_category_question}))

    if is_error(result):
        rows.append({"reply": reply, "error": f"{result.error_code}: {result.message}"})
        continue

    answer = result.answers["tone_category"]  # the answer object for our choice question

    rows.append({
        "reply": reply[:60] + ("..." if len(reply) > 60 else ""),
        "tone_category": answer.choice,                  # the category Jev picked
        "P(chosen)": round(answer.probabilities[answer.choice], 2),  # probability of that pick
        "confidence": round(answer.confidence, 2),
    })

pd.DataFrame(rows)

,reply,tone_category,P(chosen),confidence
0,Thanks for reaching out! I'm sorry about the t...,warm,0.97,0.96
1,Read the docs. This was already answered.,rude,0.94,0.92
2,Your order has shipped and should arrive Friday.,neutral,0.98,0.97


### Reading the category results

- **tone_category** is a readable label, which is handy for dashboards and routing.
- **P(chosen)** is the probability Jev gives to its own pick. A pick with a low probability is a weak decision.
- Each question type answers a different need: the yes/no gives a **decision**, the scale gives a **degree**, and the choice gives a **label**.

## Step 3.5: Ask all three in one request

Now that each question type is clear, we can send all three in **a single request**. Jev accepts a dictionary of questions, and the dictionary keys are the names we use to read each answer. This is cheaper than three separate calls.

In [ ]:
questions = {
    "polite": polite_question,
    "tone": tone_question,
    "tone_category": tone_category_question,
}

rows = []

for reply in replies:
    result = jev.evaluate(JevInput(state=reply, questions=questions))

    if is_error(result):
        rows.append({"reply": reply, "error": f"{result.error_code}: {result.message}"})
        continue

    answers = result.answers  # dictionary: question name -> answer object

    p_polite = answers["polite"].noul
    tone_score = answers["tone"].score
    tone_conf = answers["tone"].confidence
    tone_cat = answers["tone_category"].choice

    rows.append({
        "reply": reply[:60] + ("..." if len(reply) > 60 else ""),
        "P(polite)": round(p_polite, 2),
        "verdict": "POLITE" if p_polite >= THRESHOLD else "NOT POLITE",
        "tone_score": round(tone_score, 2),
        "tone_category": tone_cat,
        "tone_confidence": round(tone_conf, 2),
    })

results_df = pd.DataFrame(rows)
results_df

,reply,P(polite),verdict,tone_score,tone_category,tone_confidence
0,Thanks for reaching out! I'm sorry about the t...,0.97,POLITE,3.00,warm,1.00
1,Read the docs. This was already answered.,0.16,NOT POLITE,0.14,rude,0.86
2,Your order has shipped and should arrive Friday.,0.78,POLITE,1.28,neutral,0.72


The three signals should agree broadly: the warm reply scores high on all of them and the rude reply scores low on all of them. The neutral reply is where they add different information.

## Step 3.6: Threshold optimisation

Only after studying each question type do we tune the threshold. The model's probabilities do not change when `THRESHOLD` changes. Only the **decision rule** changes, so we can try several thresholds on the probabilities already collected in `results_df`, with no extra API calls.

In [ ]:
for t in [0.2, 0.5, 0.9]:  # a loose, a middle, and a strict threshold
    verdicts = ["POLITE" if p >= t else "NOT POLITE" for p in results_df["P(polite)"]]
    print(f"THRESHOLD = {t}: {verdicts}")

THRESHOLD = 0.2: ['POLITE', 'NOT POLITE', 'POLITE']
THRESHOLD = 0.5: ['POLITE', 'NOT POLITE', 'POLITE']
THRESHOLD = 0.9: ['POLITE', 'NOT POLITE', 'NOT POLITE']


- A **low** threshold passes almost everything, including borderline replies (more false passes).
- A **high** threshold passes only replies Jev is very sure about, so borderline replies such as the neutral one may flip to NOT POLITE (more false rejections).
- There is no universally correct threshold. You choose one by measuring against examples you have labelled (see the practice exercises).

### Check your understanding
1. What happens to the verdicts if you change `THRESHOLD` to `0.9`?
2. Why might a *neutral* factual reply get a mid-range score?

*(Answers are in the answer key at the end.)*

# **Part 2: Mock RAG with Relevance Filtering**

## What is RAG?

**Retrieval-Augmented Generation (RAG)** answers questions in two steps:
1. **Retrieve**: find document chunks that might contain the answer.
2. **Generate**: give those chunks to a language model as context and ask it to answer.

## The problem

Retrievers are imperfect. They often return chunks that *look* similar but do not help. Irrelevant chunks can **distract** the generator and cause wrong answers.

## The solution: a relevance gate

We insert a checkpoint between retrieval and generation:

```
Query -> Retrieve chunks -> Check each chunk with Jev -> Keep relevant -> Generate
```

For each chunk, Jev answers a ready-made question (`RAG_RELEVANCE`): *can this chunk answer the query?* It is a `ChoiceQuestion` with two options, **`relevant`** and **`unrelated`**, so the number we need is `probabilities["relevant"]`.

## Why "mock"?

To keep the focus on evaluation, we fake the retriever with a small hand-written knowledge base. In production you would replace it with a real vector search.

## Step 4.1: Build the mock knowledge base and retriever

In [ ]:
# A tiny "knowledge base" about a store's refund policy, with deliberate noise
KB = [
    "Refunds are issued to the original payment method within 5-7 business days.",    # relevant
    "Our headquarters is located in Austin, Texas, and has a rooftop garden.",        # noise
    "To request a refund, open Orders, select the item, and click 'Request refund'.", # relevant
    "The company was founded in 2012 by two university friends.",                     # noise
    "Items must be returned within 30 days of delivery to qualify for a refund.",     # relevant
]


def mock_retrieve(query, k=5):
    # A fake retriever: ignores the query and returns the first k chunks.
    # Real retrievers rank chunks by similarity, but they also return noise like this.
    return KB[:k]

## Step 4.2: Inspect the raw answer (good practice)

Library details can differ between versions, so it is wise to look at the **raw output** once before writing code that depends on its shape. The next cell prints exactly what Jev returns for one relevant chunk. You should see a choice-style answer with `.choice`, `.probabilities`, and `.confidence`.

In [ ]:
# A sample query and a chunk we expect to be relevant to it
q = "How do I get a refund and how long does it take?"
sample_chunk = KB[0]

# Call Jev directly so we can see the unprocessed result
raw = jev.evaluate(JevInput(
    state={"query": q, "context": sample_chunk},  # the query and the chunk to judge
    questions={"relevant": RAG_RELEVANCE},        # the ready-made relevance question
))

# Show the full answer object
print(raw if is_error(raw) else raw.answers["relevant"])

type='choice' choice='relevant' probabilities={'relevant': 0.73, 'unrelated': 0.27} confidence=0.46


## Step 4.3: Write the relevance check

This function wraps the Jev call and returns a simple decision: *keep this chunk or not*, plus the probability behind it.

Note the **fail-open** design: if the check errors, we **keep** the chunk. Silently losing real evidence is usually worse than letting one noisy chunk through.

In [ ]:
def check_relevance(query, chunk, threshold=0.5):
    # Ask Jev the ready-made RAG_RELEVANCE question about this (query, chunk) pair
    result = jev.evaluate(JevInput(
        state={"query": query, "context": chunk},  # the thing being judged
        questions={"relevant": RAG_RELEVANCE},     # the question to answer
    ))

    # If the evaluation call failed, keep the chunk ("fail open") so we never lose evidence silently
    if is_error(result):
        print(f"  [warning] {result.error_code}: {result.message}")
        return True, float("nan")

    answer = result.answers["relevant"]   # the answer object for our question
    p = answer.probabilities["relevant"]  # probability that the chunk is relevant
    return p >= threshold, p              # (keep it?, probability)

## Step 4.4: Assemble the full pipeline

Now we connect everything: retrieve, check every chunk, sort into **kept** and **dropped**, then build the prompt a generator would receive. We print each decision so the process is transparent and easy to debug.

In [ ]:
def answer_with_rag(query):
    chunks = mock_retrieve(query)   # Step 1: retrieve candidate chunks
    kept, dropped = [], []          # Step 2: we will sort chunks into two buckets

    for chunk in chunks:                               # check every retrieved chunk
        keep, p = check_relevance(query, chunk)        # ask Jev: relevant or not?
        (kept if keep else dropped).append((chunk, p)) # put it in the matching bucket

    # Report what the gate decided, so the process is transparent
    print(f"Query: {query}")
    print(f"Kept {len(kept)} chunks, dropped {len(dropped)}\n")
    for chunk, p in kept:
        print(f"  KEPT    (p={p:.2f}): {chunk}")
    for chunk, p in dropped:
        print(f"  DROPPED (p={p:.2f}): {chunk}")

    # Step 3: build the context string using ONLY the kept chunks
    context = "\n".join(f"- {c}" for c, _ in kept)

    # Step 4: build the final prompt a generator LLM would receive
    prompt = (
        "Answer using only the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}"
    )
    return prompt  # in a real system you would send this prompt to an LLM


final_prompt = answer_with_rag("How do I get a refund and how long does it take?")
print("\n--- FINAL PROMPT SENT TO THE GENERATOR ---\n")
print(final_prompt)

Query: How do I get a refund and how long does it take?
Kept 1 chunks, dropped 4

  KEPT    (p=0.69): Refunds are issued to the original payment method within 5-7 business days.
  DROPPED (p=0.00): Our headquarters is located in Austin, Texas, and has a rooftop garden.
  DROPPED (p=0.47): To request a refund, open Orders, select the item, and click 'Request refund'.
  DROPPED (p=0.00): The company was founded in 2012 by two university friends.
  DROPPED (p=0.16): Items must be returned within 30 days of delivery to qualify for a refund.

--- FINAL PROMPT SENT TO THE GENERATOR ---

Answer using only the context below.

Context:
- Refunds are issued to the original payment method within 5-7 business days.

Question: How do I get a refund and how long does it take?


**What to look for**

- The chunks about **refund steps, timing, and return windows** should be kept.
- The chunks about **headquarters** and **company history** should be dropped.
- The final prompt now contains only useful evidence, so the generator is less likely to be distracted.

**Design lessons**

- **Fail open:** if the check errors out, we keep the chunk.
- **Threshold trade-off:** a high threshold gives cleaner context but risks dropping useful chunks. A low threshold keeps more but lets noise through.
- **Transparency:** printing kept and dropped chunks makes the pipeline easy to debug.

## Step 4.5: Try a different query

The same gate should behave differently when the question changes. Run the next cell and compare which chunks are kept.

In [ ]:
# A query about a completely different topic from refunds
other_prompt = answer_with_rag("Where is the company headquartered?")

Query: Where is the company headquartered?
Kept 1 chunks, dropped 4

  KEPT    (p=1.00): Our headquarters is located in Austin, Texas, and has a rooftop garden.
  DROPPED (p=0.00): Refunds are issued to the original payment method within 5-7 business days.
  DROPPED (p=0.00): To request a refund, open Orders, select the item, and click 'Request refund'.
  DROPPED (p=0.00): The company was founded in 2012 by two university friends.
  DROPPED (p=0.00): Items must be returned within 30 days of delivery to qualify for a refund.


# **Summary**

- A **decision model** like Jev returns **typed answers with probabilities** instead of free text.
- **`NoulQuestion`** gives a yes/no probability, **`ScoreQuestion`** gives a graded level, and **`ChoiceQuestion`** picks a category from options you describe.
- Evaluation outputs are **probabilities**, so choose thresholds deliberately and handle uncertainty.
- In **RAG**, a **relevance gate** between retrieval and generation removes distracting chunks.
- Good evaluators handle errors explicitly and make their decisions visible.

# **Answer Key**

**Check your understanding (Part 1)**

1. At `0.9`, only replies where Jev is very sure of politeness stay `POLITE`. Borderline replies, often the neutral one, flip to `NOT POLITE`. A stricter threshold means fewer false passes but more false rejections.
2. A neutral factual reply contains no rude content, but it also has no courteous signals such as thanks or an apology. Jev's probability is spread across options, so you see a mid-range score and often lower confidence.

**Exercise 1 hint:** Lowering the threshold keeps more chunks (possibly noisy ones). Raising it keeps fewer (possibly dropping useful ones).